# TechHub Retail — Sales, Profitability & Customer Analytics

**Google Colab notebook | Business Intelligence / Data Analytics | 2024–2025**

> **Important:** This notebook ships with **synthetic demonstration data**, not the original data from the internship PDF. The PDF shows analyses and charts, but does **not** contain the underlying rows. Replace the sample CSVs with your real exports to reproduce your actual project results.

What this notebook does: upload datasets → validate relationships → calculate KPIs → plot sales trends, profitability, demographics, acquisition, product lifecycle and customer profit → export summary tables.


## 1. Upload your five datasets

1. Download **`TechHub_Retail_Synthetic_Datasets.zip`** alongside this notebook.
2. Run the cell below in Colab and select the ZIP file when the upload window opens.
3. The notebook reads five linked CSV files from `techhub_retail_data/`.

**Using original data?** Replace the sample files with your real CSV exports, using the field names in `README_DATASET.md`. Then rerun all cells.


In [ ]:
from pathlib import Path
import zipfile

DATA_DIR = Path('techhub_retail_data')
ARCHIVE = Path('TechHub_Retail_Synthetic_Datasets.zip')
NEEDED = ['sales.csv', 'customers.csv', 'products.csv', 'suppliers.csv', 'acquisition.csv']

if not all((DATA_DIR / name).exists() for name in NEEDED):
    if not ARCHIVE.exists():
        try:
            from google.colab import files
        except ImportError as exc:
            raise FileNotFoundError('Place the ZIP next to the notebook, or extract it into techhub_retail_data/.') from exc
        print('Please upload TechHub_Retail_Synthetic_Datasets.zip')
        uploaded = files.upload()
        if ARCHIVE.name not in uploaded:
            raise FileNotFoundError(f'Upload {ARCHIVE.name}, then rerun this cell.')
    with zipfile.ZipFile(ARCHIVE) as zf:
        zf.extractall('.')

assert all((DATA_DIR / name).exists() for name in NEEDED), 'One or more CSVs are missing.'
print('Dataset folder ready:', DATA_DIR.resolve())
for name in NEEDED:
    print(' •', name)


## 2. Load data and verify the data model

The report describes Tableau relationships through `Customer ID` and `Product ID`. These CSVs add a `Supplier ID` relationship and an acquisition table at one record per customer. Keep sales at the order-line grain to avoid duplicated revenue.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

sales = pd.read_csv(DATA_DIR / 'sales.csv', parse_dates=['order_date'])
customers = pd.read_csv(DATA_DIR / 'customers.csv', parse_dates=['signup_date'])
products = pd.read_csv(DATA_DIR / 'products.csv', parse_dates=['launch_date'])
suppliers = pd.read_csv(DATA_DIR / 'suppliers.csv')
acquisition = pd.read_csv(DATA_DIR / 'acquisition.csv', parse_dates=['acquisition_date'])

for title, df in [('Sales', sales), ('Customers',customers),('Products',products),('Suppliers',suppliers),('Acquisition',acquisition)]:
    print(f'{title:12} {len(df):>7,} rows  |  missing cells: {df.isna().sum().sum()}')

assert sales.sale_id.is_unique, 'Duplicate sale_id found'
assert customers.customer_id.is_unique, 'Duplicate customer ID found'
assert products.product_id.is_unique, 'Duplicate product ID found'
assert suppliers.supplier_id.is_unique, 'Duplicate supplier ID found'
assert acquisition.customer_id.is_unique, 'Acquisition data has repeated customer IDs'
assert set(sales.customer_id) <= set(customers.customer_id), 'Unknown customer ID in sales'
assert set(sales.product_id) <= set(products.product_id), 'Unknown product ID in sales'
assert set(products.supplier_id) <= set(suppliers.supplier_id), 'Unknown supplier ID in products'
assert set(acquisition.customer_id) == set(customers.customer_id), 'Missing customer acquisition record'

joined = (sales
    .merge(products[['product_id','product_name','category','supplier_id','launch_date']], on='product_id', how='left', validate='many_to_one')
    .merge(customers[['customer_id','age_group','region','loyalty_tier']], on='customer_id', how='left', validate='many_to_one')
    .merge(suppliers[['supplier_id','supplier_name']], on='supplier_id', how='left', validate='many_to_one')
    .merge(acquisition[['customer_id','channel','acquisition_cost_eur']], on='customer_id', how='left', validate='many_to_one')
)
assert len(joined) == len(sales), 'Joining tables changed the sales row count'
assert np.allclose(joined.revenue_eur, joined.quantity * joined.unit_price_eur * (1 - joined.discount_pct), atol=.011)
assert np.allclose(joined.profit_eur, joined.revenue_eur - joined.cost_eur, atol=.011)
joined['year'] = joined.order_date.dt.year
joined['quarter'] = joined.order_date.dt.to_period('Q').astype(str)
joined['month'] = joined.order_date.dt.to_period('M').astype(str)
joined['product_age_months'] = ((joined.order_date.dt.year - joined.launch_date.dt.year)*12
                                + joined.order_date.dt.month - joined.launch_date.dt.month)
assert (joined.product_age_months >= 0).all(), 'Sale precedes product launch'
print('\nRelationships and monetary formulas validated.')
display(joined.head(5))


## 3. Key performance indicators (KPIs)

Filter to **2025** to evaluate the business year discussed in the report. Change `ANALYSIS_YEAR` if needed. AOV is computed per **distinct order**, not per order line. 

In [ ]:
ANALYSIS_YEAR = 2025
view = joined.loc[joined.year == ANALYSIS_YEAR].copy()
if view.empty:
    raise ValueError(f'No sales found for {ANALYSIS_YEAR}. Change ANALYSIS_YEAR to a year in your data.')

def safe_div(numerator, denominator):
    return numerator / denominator if denominator else float('nan')

revenue = view.revenue_eur.sum()
profit = view.profit_eur.sum()
margin = 100 * safe_div(profit, revenue)
unique_orders = view.order_id.nunique()
unique_customers = view.customer_id.nunique()
aov = safe_div(revenue, unique_orders)

kpis = pd.DataFrame({'KPI':['Revenue','Profit','Profit margin','Distinct orders','Active customers','Average order value (AOV)'],
                     'Value':[f'€{revenue:,.2f}',f'€{profit:,.2f}',f'{margin:.1f}%',f'{unique_orders:,}',f'{unique_customers:,}',f'€{aov:,.2f}']})
print(f'TechHub Retail — synthetic dataset | {ANALYSIS_YEAR}')
display(kpis)


## 4. Revenue and profit by product category

Use the category comparison to see why **high revenue does not necessarily mean high margin**, consistent with Figure 1 in the report.

In [ ]:
category_perf = (view.groupby('category',as_index=False)
                 .agg(revenue_eur=('revenue_eur','sum'),profit_eur=('profit_eur','sum'),orders=('order_id','nunique')))
category_perf['profit_margin_pct'] = 100 * category_perf.profit_eur / category_perf.revenue_eur
category_perf = category_perf.sort_values('revenue_eur',ascending=False)
fig,axes = plt.subplots(1,2,figsize=(14,5),layout='constrained')
axes[0].barh(category_perf.category, category_perf.revenue_eur/1e6)
axes[0].invert_yaxis(); axes[0].set(xlabel='Revenue (€ million)',title='Revenue by category')
axes[1].barh(category_perf.category, category_perf.profit_margin_pct)
axes[1].invert_yaxis(); axes[1].set(xlabel='Profit margin (%)',title='Profit margin by category')
plt.show()
display(category_perf.round(2))


## 5. Sales and profit trends / seasonality

Related to Figure 3 and the report's finding that Q4 performance is strong.

In [ ]:
monthly = (view.groupby('month',as_index=False)
          .agg(revenue_eur=('revenue_eur','sum'),profit_eur=('profit_eur','sum')))
fig, ax = plt.subplots(figsize=(12,5))
ax.plot(monthly.month, monthly.revenue_eur / 1e6, marker='o', label='Revenue')
ax.plot(monthly.month, monthly.profit_eur / 1e6, marker='o', label='Profit')
ax.set(title=f'Monthly revenue and profit — {ANALYSIS_YEAR}', ylabel='€ million',xlabel='Month')
ax.tick_params(axis='x',rotation=45); ax.legend(); ax.grid(alpha=.2)
plt.show()

quarterly = (view.groupby('quarter',as_index=False).agg(revenue_eur=('revenue_eur','sum'),profit_eur=('profit_eur','sum')))
display(quarterly)


## 6. Supplier profitability

Compare suppliers on *margin*, not just revenue. Margins from very low sales volume can be misleading, so the code excludes suppliers below 1% of analyzed revenue from the chart.

In [ ]:
supplier_perf = (view.groupby('supplier_name',as_index=False)
                 .agg(revenue_eur=('revenue_eur','sum'),profit_eur=('profit_eur','sum')))
supplier_perf['profit_margin_pct'] = 100*supplier_perf.profit_eur/supplier_perf.revenue_eur
supplier_plot = supplier_perf.loc[supplier_perf.revenue_eur >= .01*revenue].sort_values('profit_margin_pct')
fig, ax = plt.subplots(figsize=(10,6))
ax.barh(supplier_plot.supplier_name,supplier_plot.profit_margin_pct)
ax.set(xlabel='Profit margin (%)', title='Supplier profitability')
plt.show()
display(supplier_perf.sort_values('profit_margin_pct',ascending=False).round(2))


## 7. Customer segments and loyalty

Segment revenue by age group and loyalty tier, and use distinct orders for segment AOV.

In [ ]:
age_perf = view.groupby('age_group',as_index=False).agg(revenue_eur=('revenue_eur','sum'),orders=('order_id','nunique'),customers=('customer_id','nunique'))
age_perf['aov_eur'] = age_perf.revenue_eur / age_perf.orders
age_order=['18-24','25-34','35-44','45-54','55+']
age_perf.age_group=pd.Categorical(age_perf.age_group,categories=age_order,ordered=True)
age_perf=age_perf.sort_values('age_group')
fig, ax=plt.subplots(figsize=(9,4))
ax.bar(age_perf.age_group.astype(str),age_perf.revenue_eur/1e6)
ax.set(title='Revenue by customer age group', ylabel='€ million',xlabel='Age group')
plt.show()
display(age_perf.round(2))

loyalty_perf = view.groupby('loyalty_tier',as_index=False).agg(revenue_eur=('revenue_eur','sum'),customers=('customer_id','nunique'))
display(loyalty_perf.sort_values('revenue_eur',ascending=False).round(2))


## 8. Acquisition channels and observed customer value (LTV proxy)

**Methodology note:** Here, "Observed LTV" is the *recorded revenue per acquired customer* over the dataset period (2024–2025), **including customers with zero recorded purchases**. It is **not a predictive lifetime value forecast**. Differences may reflect different acquisition dates and customer characteristics; do not interpret them as causal channel effects.

In [ ]:
recorded_value = sales.groupby('customer_id',as_index=False).agg(observed_revenue_eur=('revenue_eur','sum'),observed_profit_eur=('profit_eur','sum'))
value_by_customer = (acquisition[['customer_id','channel','acquisition_cost_eur']]
                     .merge(recorded_value,on='customer_id',how='left',validate='one_to_one'))
value_by_customer[['observed_revenue_eur','observed_profit_eur']] = value_by_customer[['observed_revenue_eur','observed_profit_eur']].fillna(0)
channel_perf = value_by_customer.groupby('channel',as_index=False).agg(
    acquired_customers=('customer_id','nunique'),
    avg_observed_ltv_eur=('observed_revenue_eur','mean'),
    avg_observed_profit_eur=('observed_profit_eur','mean'),
    avg_acquisition_cost_eur=('acquisition_cost_eur','mean'))
channel_perf=channel_perf.sort_values('avg_observed_ltv_eur',ascending=False)
fig,ax=plt.subplots(figsize=(11,5))
ax.bar(channel_perf.channel,channel_perf.avg_observed_ltv_eur)
ax.set(xlabel='Acquisition channel',ylabel='Observed recorded revenue / customer (€)',title='Channel comparison — observed customer value')
ax.tick_params(axis='x',rotation=40)
plt.show()
display(channel_perf.round(2))


## 9. Product age and lifecycle

The report highlights a potential decline in sales beyond 24 months. This view describes **revenue per observed product-age month**, and is not a controlled causal test: product mix, availability, and launches can influence the result.

In [ ]:
age_bins = [-1, 5, 11, 23, 35, 59, 500]
age_labels = ['0–5','6–11','12–23','24–35','36–59','60+']
view['product_age_band'] = pd.cut(view.product_age_months,bins=age_bins,labels=age_labels)
product_age = view.groupby('product_age_band',observed=True).agg(
    revenue_eur=('revenue_eur','sum'), sales_lines=('sale_id','count'),products_sold=('product_id','nunique')).reset_index()
fig,ax=plt.subplots(figsize=(9,4))
ax.bar(product_age.product_age_band.astype(str),product_age.revenue_eur/1e6)
ax.set(xlabel='Product age at purchase (months)',ylabel='Revenue (€ million)',title='Revenue by product age band')
plt.show()
display(product_age.round(2))


## 10. Revenue and profit by customer — scatterplot

Compare customer-level revenue to profit, coloring by acquisition channel (analogous to the report's Figure 5). This uses **2025** customer spend and profit.

In [ ]:
customer_profit = view.groupby(['customer_id','channel'],as_index=False).agg(revenue_eur=('revenue_eur','sum'),profit_eur=('profit_eur','sum'))
fig,ax=plt.subplots(figsize=(10,6))
for channel, subset in customer_profit.groupby('channel'):
    ax.scatter(subset.revenue_eur,subset.profit_eur,s=16,alpha=.55,label=channel)
ax.set(xlabel='Customer revenue (€)',ylabel='Customer profit (€)',title='Customer revenue vs profit')
ax.grid(alpha=.15);ax.legend(title='Acquisition channel',fontsize=8,loc='upper left')
plt.show()


## 11. Product performance: top products

This bar chart is a robust alternative to the product treemap in the report (Figure 4).

In [ ]:
top_products = (view.groupby(['product_id','product_name','category'],as_index=False)
                .agg(revenue_eur=('revenue_eur','sum'),profit_eur=('profit_eur','sum'))
                .nlargest(12,'revenue_eur').sort_values('revenue_eur'))
fig, ax=plt.subplots(figsize=(10,6))
ax.barh(top_products.product_name,top_products.revenue_eur/1000)
ax.set(xlabel='Revenue (€ thousand)',title='Top 12 products by revenue')
plt.show()
display(top_products.sort_values('revenue_eur',ascending=False).round(2))


## 12. Export analyzed tables

Save clean summary CSVs for Power BI, Tableau, Excel, or your project submission. In Google Colab, the ZIP will be downloaded automatically.

In [ ]:
from pathlib import Path
import zipfile
export_dir=Path('techhub_analysis_exports')
export_dir.mkdir(exist_ok=True)
exports={'category_performance.csv':category_perf,'monthly_performance.csv':monthly,
         'quarterly_performance.csv':quarterly,'supplier_performance.csv':supplier_perf,
         'age_group_performance.csv':age_perf,'channel_performance.csv':channel_perf,
         'product_age_performance.csv':product_age,'customer_profit.csv':customer_profit}
for filename,frame in exports.items(): frame.to_csv(export_dir/filename,index=False)
export_zip=Path('TechHub_Retail_Analysis_Exports.zip')
with zipfile.ZipFile(export_zip,'w',zipfile.ZIP_DEFLATED) as archive:
    for file in sorted(export_dir.glob('*.csv')): archive.write(file,arcname=file.name)
print('Saved', export_zip, 'with', len(exports),'CSV summaries.')
try:
    from google.colab import files
    files.download(str(export_zip))
except ImportError:
    print('Running outside Colab; use the ZIP saved in your working directory.')


## 13. Business recommendations for your submission

The original report recommends prioritizing profitable categories and suppliers, investing in high-value acquisition channels, strengthening loyalty, and aligning stock planning with Q4 demand and product aging. Use the charts above to **re-evaluate** those recommendations against your actual records.

**Submission integrity:** Do not describe the synthetic dataset or sample results as the real TechHub Retail source data. When the original exports become available, replace these files and rerun the analysis.
